In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("proj2.ipynb")

# Project 2: Query Performance
## Due Date: Wednesday, October 7, 5:00 PM

## Assignment Details
In this project, we will explore how the database system optimizes query execution and how users can further tune the performance of their queries.

This project works with the [Lahman's Baseball Database](http://seanlahman.com/), an open source collection of baseball statistics from 1871 to 2020. It contains a variety of data, like batting statistics, team stats, managerial records, Hall of Fame records, and much more.

You may find this project's queries to be simpler than in Project 1. However, although the queries may not be as complex, we still expect you to spend ample time thinking through the effects of each of the methods, as reasoning about the tradeoff between different approaches is the goal of this assignment.

**Note:** If at any point during the project, the internal state of the database or its tables have been modified in an undesirable way (i.e. a modification not resulting from the instructions of a question), restart your kernel and clear output and simply re-run the notebook as normal. This will shutdown your current connection to the database, which will prevent the issue of multiple connections to the database at any given point, and when re-running the notebook you will create a fresh database based on the provided Postgres dump.

## Logistics & Scoring Breakdown

- Unless explicitly stated otherwise, each coding question has **both public tests and hidden tests**. Roughly 50% of your coding grade will be made up of your score on the public tests released to you, while the remaining 50% will be made up of unreleased hidden tests.
- Public tests for multiple choice questions are for sanity check only (e.g. you are answering in the correct format). Partial credit will be awarded.
- Free-response questions (marked 'm' in the table below) will be manually graded. Please answer thoughtfully and concisely in complete sentences, drawing from knowledge in lectures and from your inspection of query plans.

We recommend that you complete the project in numerical order, but do not require you to do so. However, subparts of a question often build on each other and should be completed in sequence. Another exception is Question 9, which changes the `salaries` table such that running older questions (Question 7aii, in particular) produces incorrect results if question 9 has already been run. In this case, please reset your connection to the SQL server manually (see point 8 of the [Assignment Tips](https://data101.org/fa25/assignment-tips/#help-why-is-my-datahub-so-slow) page for instructions on how to restart you DataHub server).

<!-- [spreadsheet](https://docs.google.com/spreadsheets/d/1lDi5AU7t12winGXEicMSyCidbtm5_xkUvAMX8D5jo-Q/edit?usp=sharing) for the points breakdown.-->

| Question  | Points |
| --------- | ------ |
| 0         | 1      |
| 1a        | 1      |
| 1bi       | 1      |
| 1bii      | 2      |
| 2a        | 1      |
| 2bi       | 1      |
| 2bii      | 1      |
| 2biii     | 1      |
| 2c        | 1      |
| 2di       | 2      |
| 2dii      | m:3    |
| 3a        | 1      |
| 3b        | 1      |
| 3c        | m:2    |
| 3d        | m:2    |
| 4a        | 1      |
| 4b        | 1      |
| 4c        | 1      |
| 4d        | m:3    |
| 5a        | 2      |
| 5b        | 1      |
| 5c        | 1      |
| 5d        | 2      |
| 5di       | m:3    |
| 6ai       | 1      |
| 6aii      | 1      |
| 6bi       | 1      |
| 6bii      | 1      |
| 6c        | 1      |
| 6d        | 1      |
| 6e        | 2      |
| 6ei       | m:3    |
| 7ai       | 1      |
| 7aii      | 1      |
| 7b        | 1      |
| 7c        | m:2    |
| 8ai       | 0.5    |
| 8aii      | 0.5    |
| 8b        | 1      |
| 8c        | 1      |
| 8d        | 2      |
| 8di       | m:3    |
| 9a        | 1      |
| 9b        | 1      |
| 9c        | m:2    |
| 10        | m:6    |
| **Total** | **70** |


**Grand Total:** 70 points (autograded: 41, manual: 29)

## Collaboration and Inegrity

This is an **individual project**. However, you’re welcome to collaborate with any other student in the class as long as it’s within the [academic honesty guidelines](https://data101.org/fa25/syllabus/#collaboration-and-integrity).

**We ask that you list any collaborators and outside sources in the cells below.** You are expected to fill this out honestly.

### Collaborators

Please include the first and last names of the other Data 101 students you work with on this project below.
- ex. Pranav Perumandla
- ...

### Outside Sources

Please include the names and urls of any outside sources you reference for this project below.
-  ex. ["How do nested loop, hash, and merge joins work? Databases for Developers Performance #7" by The Magic of SQL](https://www.youtube.com/watch?v=pJWCwfv983Q)
- ...

In [2]:
# Run this cell to set up imports
import numpy as np
import pandas as pd

## Getting Connected
Similar to Project 1, we will be using the `JupySQL` library to connect this notebook to a PostgreSQL database server on your JupyterHub account. Run the following cell to initiate the connection.

In [3]:
%reload_ext sql
%sql postgresql://jovyan@127.0.0.1:5432/postgres

Connecting to 'postgresql://jovyan@127.0.0.1:5432/postgres'

In [4]:
# See full display
%config SqlMagic.displaylimit = 50

## Setting up the Database
The following cell will create the `baseball` database (if needed), unzip the Postgres dump of the Lahman's Baseball Database and populate the `baseball` database with the desired tables and data.

**Note:** If you run into the **role does not exist**/**database does not exist** error the first time you run this cell, feel free to ignore it. It does not affect data import.

In [5]:
!unzip -u data/baseball.zip -d data/

Archive:  data/baseball.zip


In [6]:
!psql postgresql://jovyan@127.0.0.1:5432/baseball -c 'SELECT pg_terminate_backend(pg_stat_activity.pid) FROM pg_stat_activity WHERE datname = current_database()  AND pid <> pg_backend_pid();'
!psql -h localhost -c 'DROP DATABASE IF EXISTS baseball'
!psql -h localhost -c 'CREATE DATABASE baseball'
!psql -h localhost -d baseball -f data/baseball.sql # dev path
# !psql -h localhost -d baseball -f ../../../_shared/data101-readonly/proj2_data/baseball.sql # Student Path
!psql -h localhost -c 'SET max_parallel_workers_per_gather = 0;'

 pg_terminate_backend 
----------------------
 t
(1 row)

DROP DATABASE
CREATE DATABASE
SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
SET
SET
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
COPY 5219
COPY 104256
COPY 179
COPY 6236
COPY 425
COPY 6879
COPY 104324
COPY 13943
COPY 17350
COPY 138838
COPY 12028
COPY 31955
COPY 13110
COPY 4191
COPY 3040
COPY 3469
COPY 93
COPY 252
COPY 19370
COPY 45806
COPY 5445
COPY 26428
COPY 1207
COPY 325
COPY 2865
COPY 120
COPY 52
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
A

Now, run the following cell to connect to the `baseball` database. There should be no errors after running the following cell.

In [7]:
%sql postgresql://jovyan@127.0.0.1:5432/baseball

Connecting and switching to connection 'postgresql://jovyan@127.0.0.1:5432/baseball'

To ensure that the connection to the database has been established, let's try grabbing the first 5 rows from the `hall_of_fame` table.

In [8]:
%%sql
SELECT * FROM hall_of_fame LIMIT 5;

Running query in 'postgresql://jovyan@127.0.0.1:5432/baseball'

5 rows affected.

player_id,year_id,voted_by,ballots,needed,votes,inducted,category,needed_note
cobbty01,1936,BBWAA,226,170,222,Y,Player,None
ruthba01,1936,BBWAA,226,170,215,Y,Player,None
wagneho01,1936,BBWAA,226,170,215,Y,Player,None
mathech01,1936,BBWAA,226,170,205,Y,Player,None
johnswa01,1936,BBWAA,226,170,189,Y,Player,None


## Connect to the grader

Run the following cell for grading purposes.

In [ ]:
# Just run the following cell, no further action is needed.
from data101_utils import GradingUtil
grading_util = GradingUtil("proj2")
grading_util.prepare_autograder()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## Table Descriptions

In its entirety the [Lahman's Baseball Database](http://seanlahman.com/) contains 27 tables containing a variety of statistics for players, teams, games, schools, etc. For simplicity, this project will focus on a subset of the tables:

* `appearances`: details on the positions each player appeared at
* `batting`: batting statistics for each player
* `college_playing`: list of players and the colleges they attended
* `hall_of_fame`: Hall of Fame voting data
* `people`: player information (name, date of birth, and biographical info)
* `salaries`: player salary data
* `schools`: list of colleges that players attended

As a reminder from Project 1, the psql command `\d <table_name>` is helpful for identifying the schema of a table.

We **highly** encourage you open a Terminal in JupyterLab and connect to the database directly. The following command will drop you in a SQL interpreter. Remember you can use `\?` for help!

```sh
psql -h localhost -d baseball
```




You may wish to review:
* The [Assignment Tips Guide](https://data101.org/sp25/assignment-tips/)
* The [Course Notes SQL Style Guide](https://data101.org/notes/appendix/sql-style.html)

For example the command below allows you to view the schema of the `people` table:

In [ ]:
!psql -h localhost -d baseball -c '\d people';

## A Bit About Baseball

Intricate knowledge of baseball, baseball history, etc. is **not** required for this project. But a basic understanding of the game can be useful. If you'd like, you can check out the [Wikipedia entry for Major League Baseball](https://en.wikipedia.org/wiki/Major_League_Baseball). In practice, we'd encourage you to avoid coming to conclusions nor doing deep analysis with datasets where you don't have the full context, but for this assignment, the details of the sport are not the goal.

You can also read more about the [Baseball Hall of Fame](https://en.wikipedia.org/wiki/List_of_members_of_the_Baseball_Hall_of_Fame). Baseball players who excel in the sport can be inducted into the Hall of Fame.

Baseball, since its inception in the late 19th century, has had an incredible track record of keeping very detailed statistics about even minor aspects of the game. This makes it a fun study for a budding (or well-experienced) data scientist.

**The Briefest Background:**
The game of baseball is played over 9 innings, between an 'offense' (a batter, and folks running around the bases) and 'defense' (a pitcher, catcher, basemen, and others). The objective, like most sportsball games, is to score the most points -- in this case, by the batter hitting the ball and running around the bases. 
In each game, both the offense an defense have a chance to bat, where players get an "at bat" (`ab`) in the database. Each 'at bat' results in the player hitting a home run (and scoring), getting "on base" (where they can later try to run to home plate), or getting "out". After the offense gets 3 outs, their turn to score is over until the next inning. (There are many more rules than these... but that's enough to make sense of the data.)

### A Bit About the Database

To be quite honest, the schema of this database isn't our favorite. It is full of abbreivated column names, mixing data types, and seldom enforces primary or foreign key relationships. The horrors!! Nevertheless, real life ain't perfect, so we'll continue to learn how to explore and get comfortable with unknown schemas. Here are a few general tips:

* While foreign keys aren't enforced, anything with an `_id` will have consistent values. (i.e. You can treat these like primary or foreign keys on tables.)
* `year_id` is called an "id", but it is merely the calendar year in which an event occured.
* Many of the game related attributes are ruthlessly abbreviated:
    * `ab` is for _at bats_
    * `g_` is for _game..._
    * `hof` is for _hall of fame_
    * Other examples (you don't need to know these, but may come across them):
    * `lg` is for _league_
    * `gp` is games played
    * `1b`, `2b`, `3b`, `hb` refer to 1st, 2nd, 3rd and home base
* In the hall of fame table `inducted` is a string value where `Y` means true
     * ...but of course, you know that this really ought to be a boolean... 

Optionally: There is a [handy guide to an R package](https://cran.r-project.org/web/packages/Lahman/Lahman.pdf) of the Lahman Basebase db. Some of the table and column names are styled slightly differently, but this may provide additional context.

---

## Navigating the Notebook

This notebook is very long! We **highly recommend using the table of contents feature** by clicking on the button with 3 dots and lines on the left sidebar.

<br>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 0: PostgreSQL Explain Analyze

<div class="alert alert-block alert-danger">
Please read through this section carefully, as a vast majority of the project will require you to inspect query plans via interpreting the output of the <code>EXPLAIN ANALYZE</code> command.
</div>

Read through the following articles to see how you can interpret the output of `EXPLAIN ANALYZE`:

1. Everything before "Tools to interpret `EXPLAIN ANALYZE` output" in [this article](https://www.cybertec-postgresql.com/en/how-to-interpret-postgresql-explain-analyze-output/)
2. PostgreSQL [documentation 14.1.2](https://www.postgresql.org/docs/current/using-explain.html#USING-EXPLAIN-ANALYZE)

<div class="alert alert-block alert-info">    
<b>Here are some key things to note for all question parts:</b>
<ul>
<li>When we ask you to identify the <b>query cost</b>, we are looking for the <b>total cost</b>.</li>
    <ul>
    <li>There are two cost values: the first is the <b>startup cost</b> (cost to return the first row) and the second is the <b>total cost</b> (cost to return all rows).</li>
    <li>The unit for the estimated query cost is an arbitrary estimation of disk I/O (1 is the cost for reading an 8kB page during a sequential scan).</li>
        <li>Feel free to round the query cost / time to the nearest integer, but we'll accept anything more exact.</li>
    </ul>
<li>When we ask you to identify the <b>query time</b>, we are looking for the <b>execution time</b> (in ms).</li>
    <ul>
        <li>We recognize that the execution time may vary between different cell executions, so the autograder will tolerate a reasonable range.</li>
    </ul>
</ul>
</div>

Now, inspect the query plan above by following the below steps:

1. Run the query below (it's the same one from the screenshot).

In [ ]:
%%sql --save query_0 result_0 <<
SELECT *
FROM people AS p
INNER JOIN college_playing AS cp
ON p.player_id = cp.player_id;

2. Run the below cell to cache the query and view the first 3 rows.

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_0 = %sqlcmd snippets query_0
grading_util.save_results("result_0", query_0, result_0);
result_0.DataFrame().head(3)

3. Run the below cell to `EXPLAIN ANALYZE` the saved query. **For this entire project when running `EXPLAIN ANALYZE`, you may ignore the "unsupported syntax" error message if it appears.**

    **Note**: If your EXPLAIN ANALYZE says that your query is doing a Merge Join, please rerun this cell 1 to 2 times until it does a Hash Join.

In [ ]:
!psql -h localhost -d baseball -c 'EXPLAIN ANALYZE {query_0}'

4. Finally, record the query **cost** and **time** for the sample query. **There are no hidden tests for Question 0.** For all questions within this project, we will accept a range of values for your costs and timings. If there are minor variations in the values as you re-run your cells, we will accept either one.

In [ ]:
sample_query_cost = ...
sample_query_timing = ...

In [ ]:
grader.check("q0")

<br/><br/>
<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 1: Queries and Views, Part 1

In Questions 1 and 2, you will compare and contrast writing queries with subqueries and views.

## Question 1a
Write a query that finds `people.name_first`, `people.name_last`, `people.player_id` and `hall_of_fame.year_id` of all people who were successfully inducted into the Hall of Fame.

**Note**: Your query should **NOT** use any sub-queries. This is what your table header should look like:

| name_first | name_last | player_id | year_id |
| --- | --- | --- | --- |

In [ ]:
%%sql --save query_1a result_1a <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_1a = %sqlcmd snippets query_1a
grading_util.save_results("result_1a", query_1a, result_1a)
result_1a.DataFrame().head(3)

In [ ]:
grader.check("q1a")

<br><br>

---

## Question 1b
In this question, we will compare the query you wrote in Question 1a against the provided query below in Question 1bi by inspecting both query plans.

### Question 1bi
Inspect the query plan for `provided_query` and the query you wrote in Question 1a by running the cells below.


In [ ]:
%%sql --save provided_query provided_result <<
-- just run this cell
SELECT name_first, name_last, p.player_id, year_id
FROM people AS p,
(
  SELECT * FROM hall_of_fame WHERE inducted = 'Y'
) AS hof
WHERE p.player_id = hof.player_id;

In [ ]:
# just run this cell
provided_query = %sqlcmd snippets provided_query
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {provided_query}"

In [ ]:
# just run this cell
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_1a}"

Record the **execution time** and **cost** for each query. Due to the fickle nature of query execution, please run `EXPLAIN ANALYZE` **multiple times** until the query plan converges to a consistent plan.

In [ ]:
provided_query_cost = ...
provided_query_timing = ...
your_query_cost = ...
your_query_timing = ...

In [ ]:
grader.check("q1bi")


### Question 1bii

Given your findings from inspecting the query plans of the two queries, consider the following statements.

**Assign the variable `q1bii` to a list of _ALL_ of the below statements that are true.**


Consider the following statements:
<br>
A. Both the queries have the same cost.
<br>
B. The provided query has a faster execution time because it makes use of a subquery.
<br>
C. The query you wrote has a faster execution time because it does not make use a subquery.
<br>
D. The provided query has less cost because it makes use of a subquery.
<br>
E. The query you wrote has less cost because it does not make use a subquery.
<br>
F. The queries have the same output.
<br>
G. The queries do not have the same output.
    
**Note:** Your answer should have the format like this if you think A and B are both true: `q1bii = ['A', 'B']`. The autograder is **case sensitive** but order should not matter.

In [ ]:
q1bii = ...

In [ ]:
grader.check("q1bii")

<br/><br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


# Question 2: Queries and Views, Part 2

In this question, you will continue analyzing queries with/without views and materialized views.

<br/><br/>

---

## Question 2a

Write a query that returns the people who were successfully inducted into the Hall of Fame and played in college at a school located in California. For each player, return their `name_first`, `name_last`, `player_id` and `school_id`, along with the `year_id` the year they were inducted into the Hall of Fame. Order by the `year_id` (ascending) and break ties on `player_id` (ascending). This is what your table header should look like:

| name_first | name_last | player_id | school_id | year_id |
| --- | --- | --- | --- | --- |

**Notes**: 
- **Do NOT use any views, materialized views, CTEs, or subqueries**
- For the baseball fanatics, FYI this dataset does not include *all* hall of fame inductees in existence and may be missing some. However, this should not affect your result; please query from the tables as given.
- You may see duplicate rows for the same player. This is expected as some players have multiple entries in the `college_playing` table for different years or schools attended. Each row in your output should represent one college attendance record for an inducted player.

In [ ]:
%%sql --save query_2a result_2a <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_2a = %sqlcmd snippets query_2a
grading_util.save_results("result_2a", query_2a, result_2a)
result_2a

In [ ]:
grader.check("q2a")

<br/><br/>

---

## Question 2 View and MV Creation

We are now going to use the query you wrote in the previous part to generate a view, called `inducted_hof_ca`, and a materialized view, `inducted_hof_ca_mat`.

**Run the below cells. You do not need to do anything more for this part.**

Note: The semicolon strip in the `EXPLAIN ANALYZE` cell is to avoid executing an empty query with double-semicolons, which causes an error.

In [ ]:
%%sql
/* just run this cell */
DROP VIEW IF EXISTS inducted_hof_ca;
CREATE VIEW inducted_hof_ca AS {{query_2a.strip(';')}};

SELECT * FROM inducted_hof_ca;

In [ ]:
%%sql
/* just run this cell */
DROP MATERIALIZED VIEW IF EXISTS inducted_hof_ca_mat;
CREATE MATERIALIZED VIEW inducted_hof_ca_mat AS {{query_2a.strip(';')}};

SELECT * FROM inducted_hof_ca_mat;

<br/><br/>

---
## Question 2b

For this question, we want to compute the count of players who were inducted into the Hall of Fame and played baseball at a college in California for each `school_id` and `year_id` combination ordered by ascending `year_id`. **Note:** For this query, `year_id` continues to refer to each player's year of induction into the Hall of Fame.

You should write three queries that accomplish this task, but with different strategies:
* Question 2bi: Use the `inducted_hof_ca` view;
* Question 2bii Use the `inducted_hof_ca_mat` view; and
* Question 2biii: **Do not use `inducted_hof_ca` view, `inducted_hof_ca_mat` materialized view, any common table expressions (CTEs), nor any subqueries.**

For all subparts in Q2b, your table header should look like this:

| school_id | year_id | count |
| --- | --- | ---|

### Question 2bi

Write a query to accomplish the task above using the `inducted_hof_ca` view.

In [ ]:
%%sql --save query_2bi result_2bi <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_2bi = %sqlcmd snippets query_2bi
grading_util.save_results("result_2bi", query_2bi, result_2bi)
result_2bi

In [ ]:
grader.check("q2bi")

<br/><br/>

### Question 2bii

Now, write the query a second time using the materialized view `inducted_hof_ca_mat`.

In [ ]:
%%sql --save query_2bii result_2bii <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_2bii = %sqlcmd snippets query_2bii
grading_util.save_results("result_2bii", query_2bii, result_2bii)
result_2bii

In [ ]:
grader.check("q2bii")

<br/><br/>

### Question 2biii

Finally, write the query a third time. Do **NOT** use the `inducted_hof_ca` view, nor the `inducted_hof_ca_mat` materialized view, nor any common table expressions (CTEs), nor any subqueries.

In [ ]:
%%sql --save query_2biii result_2biii <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_2biii = %sqlcmd snippets query_2biii
grading_util.save_results("result_2biii", query_2biii, result_2biii)
result_2biii

In [ ]:
grader.check("q2biii")

<br/><br/>

---

## Question 2c
Inspect the query plans for the three queries you wrote above by running the following cells.

There are no hidden tests for Question 2c.

In [ ]:
# just run this cell
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_2bi}";

In [ ]:
# just run this cell
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_2bii}";

In [ ]:
# just run this cell
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_2biii}";

Then, record the execution time and cost for each query.

In [ ]:
with_view_cost = ...
with_view_timing = ...
with_materialized_view_cost = ...
with_materialized_view_timing = ...
without_view_cost = ...
without_view_timing = ...

In [ ]:
grader.check("q2c")

<br/><br/>

---

## Question 2d

Given your findings from inspecting the query plans in this question, as well as your understanding of views and materialized views from lectures, discuss the takeaways of using views and materialized views.

### Question 2di

**Assign the variable `q2di` to a list of _ALL_ of the below statements that are true.**

A. Views will reduce the execution time and the cost of a query.<br/>
B. Views will reduce the execution time of a query, but not the cost.<br/>
C. Views will reduce the cost of a query, but not the execution time.<br/>
D. Materialized views reduce the execution time and the cost of a query.<br/>
E. Materialized views reduce the execution time, but not cost of a query.<br/>
F. Materialized views reduce the cost of a query, but not the execution time.<br/>
G. Materialized views will result in the same query plan as a query using views.<br/>
H. Materialized views and views take the same time to create for the same query.<br/>
I. Materialized views take less time to create than a view for the same query.<br/>
J. Materialized views take more time to create than a view for the same query.<br/>
    
**Note:** Your answer should have the format like this if you think A and B are both true: `q2di = ['A', 'B']`. The autograder is **case sensitive** but order should not matter.

In [ ]:
q2di = ...

In [ ]:
grader.check("q2di")

<!-- BEGIN QUESTION -->

### Question 2dii

1. **Explain your answer** to the previous part (Question 2di) based on your understanding of both the theoretical aspects of query planning and optimization discussed in class, as well as the EXPLAIN ANALYZE results - but bear in mind that sometimes EXPLAIN ANALYZE results may be unpredictable/hard to explain (because of contention, caching, noise/randomness).
2. If there were any options you did NOT select, **choose any one of them and explain why you did not select it.**

Please limit your sentence to **100 words (~5 sentences)** and **explicitly state which answer option(s) you chose or didn't choose** in addition to your explanations. For example, you could write, "I chose (A) because..." or "I did NOT choose (B) because..."

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 3: Predicate Pushdown
In this question, we will explore the impact of predicates (i.e., filters) on a query's execution, particularly inspecting when the optimizer applies predicates.


## Question 3a

Recall the `inducted_hof_ca` view created in `Question 2`. Run `EXPLAIN ANALYZE` for a query that that gets all rows from the view, and record the execution time and cost.

Note: You might see warnings like "...contains unsupported syntax. Falling back..." which you can safely ignore as long as you see the `EXPLAIN ANALYZE` output.

In [ ]:
%%sql
...
;

In [ ]:
query_view_cost = ...
query_view_timing = ...

In [ ]:
grader.check("q3a")

<br><br>

---

## Question 3b

Now, run `EXPLAIN ANALYZE` on the same query as 3a, except now add a filter to only return rows from `inducted_hof_ca` where the year is after 2010. Inspect the query plan and record the execution time and cost.

There are no hidden tests for Question 3b.

Note: You might see warnings like "...contains unsupported syntax. Falling back..." which you can safely ignore as long as you see the `EXPLAIN ANALYZE` output.

In [ ]:
%%sql
...
;

In [ ]:
query_view_with_filter_cost = ...
query_view_with_filter_timing = ...

In [ ]:
grader.check("q3b")

<!-- BEGIN QUESTION -->

## Question 3c

Given your findings from inspecting the query plans of queries from Questions 3a and 3b, fill in the blank and **justify your answer**. Explain your answer based on your understanding of both the theoretical aspects of query planning and optimization discussed in class, as well as the EXPLAIN ANALYZE results. Please limit your response to 60 words (~3 sentences).

**Note:** Your answer should be formatted as follows if you think option A is true: `(A) because ...`

**Adding a filter ___ the cost.**
<br>
A. increased
<br>
B. decreased
<br>
C. did not change

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br><br>

---

## Question 3d

Given your findings from inspecting the query plans of queries from Questions 3a and 3b, fill in the blank and **justify your answer**. Explain your answer based on your understanding of both the theoretical aspects of query planning and optimization discussed in class, as well as the EXPLAIN ANALYZE results. Please limit your response to **60 words (~3 sentences)**.

**Note:** Your answer should be formatted as follows if you think option A is true: `(A) because ...`

**Adding a filter ___ the execution time.**
<br>
A. increased
<br>
B. decreased
<br>
C. did not change

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br><br>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 4: Join Approaches

In this question, we'll explore different join approaches (Nested Loop Join, Merge Join, Hash Join) and discuss how the query optimizer picks the best approach.

<br/><br/>

---

## Question 4a
Perform an inner join on the `people` and `college_playing` tables on the `player_id` column. Select all columns. (No need to run `EXPLAIN ANALYZE` here, we do it for you below.)

In [ ]:
%%sql --save query_4a result_4a <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_4a = %sqlcmd snippets query_4a
grading_util.save_results("result_4a", query_4a, result_4a);

display(result_4a.DataFrame().head(3))

!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_4a}";

If you haven't already, run the cell above to inspect the query plan for your command.

**Which join approach did the query optimizer choose?** 

A. Nested Loop Join<br/>
B. Merge Join<br/>
C. Hash Join<br/>
D. None of the Above

Assign the variable `q4a` to the correct letter choice above, e.g., `q4a = 'A'`.

In [ ]:
q4a = ...

In [ ]:
grader.check("q4a")

<br><br>

---

## Question 4b

Perform the same query in Question 4a, but now also **sort the output by `player_id`**.

In [ ]:
%%sql --save query_4b result_4b <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_4b = %sqlcmd snippets query_4b
grading_util.save_results("result_4b", query_4b, result_4b);

display(result_4b.DataFrame().head(3))

!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_4b}";

If you haven't already, run the cell above to inspect the query plan for your command.

**Which join approach did the query optimizer choose?** 

A. Nested Loop Join<br/>
B. Merge Join<br/>
C. Hash Join<br/>
D. None of the Above

Assign the variable `q4b` to the correct letter choice above, e.g., `q4b = 'A'`.

In [ ]:
q4b = ...

In [ ]:
grader.check("q4b")

<br><br>

---
## Question 4c
Write a query to retrieve all possible player pair permutations (e.g. `(Player 1, Player 2)` is treated as a different pairing from `(Player 2, Player 1)` because the order matters). Select all columns, but **limit to 1000 rows** to ensure your query doesn't take an exorbitant amount of time to run.

**Hint:** You can do this by performing an inner join of the `people` table on itself with an inequality condition.

In [ ]:
%%sql --save query_4c result_4c <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_4c = %sqlcmd snippets query_4c
grading_util.save_results("result_4c", query_4c, result_4c);

display(result_4c.DataFrame().head(3))

!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_4c}";

If you haven't already, run the cell above to inspect the query plan for your command.

**Which join approach did the query optimizer choose?** 

A. Nested Loop Join<br/>
B. Merge Join<br/>
C. Hash Join<br/>
D. None of the Above

Assign the variable `q4c` to the correct letter choice above, e.g., `q4c = 'A'`.

In [ ]:
q4c = ...

In [ ]:
grader.check("q4c")

<!-- BEGIN QUESTION -->

<br><br>

---
## Question 4d

Given your findings above, why did the query optimizer ultimately choose the specific join approach you found in each of the above three scenarios in Questions 4a, 4b, and 4c?

If you feel stuck, here are some things to consider: Does a non-equijoin constrain us to certain join approaches? What's an added benefit in regards to the output of merge join? How does table size affect which join is used?

**Note:**
- Restate your answer for each of the subparts. Please limit your answer to at most 100 words (~5 sentences). Your answer should be formatted as follows: 
```
Q4a: (A) because ...
Q4b: (A) because ...
Q4c: (A) because ...
```

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/><br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 5: Indexes, Part 1

In Questions 5, 6, and 7, you will analyze how indexes impact query performance.

<br/>

---

## Question 5a
Write a SQL query that considers only players who appeared in **exactly 10 games** in a given year. For each `player_id`, return the following:

- `average_salary`: the player’s average salary across all qualifying years,

- `first_year`: the first year the player appeared,

- `last_year`: the most recent year the player appeared,

- `num_years`: the total number of years in which the player appeared.

**Hints:**
- The number of games in which a player batted can be found in the `g_batting` column of the `appearances` table. 
- Your query should join the `salaries` and `appearances` table on all the common columns `year_id`, `team_id`, and `player_id`, so feel free to use a natural join. 

Your table header should look like this:

| player_id | average_salary | first_year | last_year | num_years |
| --- | --- | --- | --- | --- |

In [ ]:
%%sql
select *
from appearances
limit 10;

In [ ]:
%%sql
/* Make sure to run this cell if you have already run Question 5b and/or 5c 
    and have not restarted your kernel since */
    
DROP INDEX IF EXISTS appearances_g_batting_idx;
DROP INDEX IF EXISTS salary_idx;

In [ ]:
%%sql --save query_5a result_5a <<
...

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_5a = %sqlcmd snippets query_5a
grading_util.save_results("result_5a", query_5a, result_5a);

display(result_5a.DataFrame().head(3))

!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_5a}";

Inspect the query plan above and record the execution time and cost.

In [ ]:
result_5a_cost = ...
result_5a_timing = ...

In [ ]:
grader.check("q5a")

<br><br>

---
## Question 5b

Add an index with name `appearances_g_batting_idx` on the `g_batting` column of the `appearances` table.

In [ ]:
%%sql
DROP INDEX IF EXISTS appearances_g_batting_idx;
DROP INDEX IF EXISTS salary_idx;
...
;

Now, re-inspect the query plan of the query from `Question 5a` and record its execution time and cost.

In [ ]:
# just run this cell
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_5a}";

In [ ]:
result_5b_cost = ...
result_5b_timing = ...

In [ ]:
grader.check("q5b")

<br><br>

---
## Question 5c

Write a query to add an index with name `salary_idx` on the `salary` column of the `salaries` table. Make sure to drop the previous index in Question 5b first (which we've provided in the starter code)!

In [ ]:
%%sql
DROP INDEX IF EXISTS appearances_g_batting_idx;
DROP INDEX IF EXISTS salary_idx;
...

Now, re-inspect the query plan of the query from Question 5a and record its execution time and cost.

Note: You might see warnings like "...contains unsupported syntax. Falling back..." which you can safely ignore as long as you see the `EXPLAIN ANALYZE` output.

In [ ]:
# just run this cell
%sql EXPLAIN ANALYZE {{query_5a}}

In [ ]:
result_5c_cost = ...
result_5c_timing = ...

In [ ]:
grader.check("q5c")

<br><br>

---

## Question 5d

Given your findings from inspecting the query plans with no indexes (Question 5a), an index on `g_batting` (Question 5b), and an index on `salary` (Question 5c), assign the variable `q5d` to a list of **ALL** of the below statements that are true.

A. Adding the `appearances_g_batting` index did not have a significant impact on the query execution time and cost.<br/>
B. Adding the `appearances_g_batting` index did have a significant impact on the query execution time, but not the cost.<br/>
C. Adding the `appearances_g_batting` index did have a significant impact on the query cost, but not the execution time.<br/>
D. Adding the `appearances_g_batting` index did have a significant impact on the query cost and execution time.<br/>
E. Adding the `salary_idx` index did not have a significant impact on the query execution time and cost.<br/>
F. Adding the `salary_idx` index did have a significant impact on the query execution time, but not the cost.<br/>
G. Adding the `salary_idx` index did have a significant impact on the query cost, but not the execution time.<br/>
H. Adding the `salary_idx` index did have a significant impact on the query cost and execution time.

**Note:** Your answer should have the format like this if you think A and B are both true: `q5b = ['A', 'B']`. The autograder is **case sensitive** but order should not matter.

In [ ]:
q5d = ...

In [ ]:
grader.check("q5d")

<!-- BEGIN QUESTION -->

### Question 5di Justification

1. **Explain your answer to Question 5d** above based on your understanding of both the theoretical aspects of query planning and optimization discussed in class, as well as the EXPLAIN ANALYZE results - but bear in mind that sometimes EXPLAIN ANALYZE results may be unpredictable/hard to explain (because of contention, caching, noise/randomness).
2. If there were any options you did NOT select, **choose any one of them and explain why you did not select it.**

Please limit your answer to 60 words (~3 sentences). You are required to **explicitly state which answer option(s) you chose or didn't choose** in addition to your explanations. For example, you could write, "I chose (A) because..." or "I did NOT choose (B) because..."

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/><br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 6: Indexes, Part 2

Continue the analysis on how indexes impact query peformance.

Before continuing, make sure to drop all existing indexes from previous questions.

In [ ]:
# you must run this cell!!!
%sql DROP INDEX IF EXISTS appearances_batting_idx;
%sql DROP INDEX IF EXISTS salary_idx;

<br><br>

---

## Question 6a

### Question 6ai

Write a query that finds the `player_id`, `year_id`, and `salary` for each player that had played 10 exactly games **and** batted in exactly 10 games (the number of games in which a player played can be found in the `g_all` column of the `appearances` table and the number of games in which a player batted can be found in the `g_batting` column of the `appearances` table). Your query should join the `salaries` and `appearances` table on all the common columns `year_id`, `team_id`, and `player_id`, so feel free to use a natural join.

Your table header should look like this:

| player_id | year_id | salary |
| --- | --- | --- |

In [ ]:
%%sql
/* Make sure to run this cell if you have already run Question 6c and/or 6d 
    and have not restarted your kernel since */
    
DROP INDEX IF EXISTS appearances_g_batting_idx;
DROP INDEX IF EXISTS appearances_g_batting_all_idx;

In [ ]:
%%sql --save query_6ai result_6ai <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_6ai = %sqlcmd snippets query_6ai
grading_util.save_results("result_6ai", query_6ai, result_6ai);

result_6ai.DataFrame().head(3)

In [ ]:
grader.check("q6ai")

### Question 6aii

Inspect the query plan and record the execution time and cost.

In [ ]:
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_6ai}";

In [ ]:
result_6aii_cost = ...
result_6aii_timing = ...

In [ ]:
grader.check("6aii")

## Question 6b

### Question 6bi

Write a query that finds the `player_id`, `year_id`, and `salary` for each player that had played exactly 10 games __or__ batted in exactly 10 games. Feel free to use a natural join to join the `salaries` and `appearances` table on all the common columns `year_id`, `team_id`, and `player_id`.

Your table header should look like this:

| player_id | year_id | salary |
| --- | --- | --- |

In [ ]:
%%sql
/* Make sure to run this cell if you have already run Question 6c and/or 6d 
    and have not restarted your kernel since */
    
DROP INDEX IF EXISTS appearances_g_batting_idx;
DROP INDEX IF EXISTS appearances_g_batting_all_idx;

In [ ]:
%%sql --save query_6bi result_6bi <<
...

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_6bi = %sqlcmd snippets query_6bi
grading_util.save_results("result_6bi", query_6bi, result_6bi);
result_6bi.DataFrame().head(3)

In [ ]:
grader.check("q6bi")

### Question 6bii

Inspect the query plan and record the execution time and cost.

In [ ]:
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_6bi}";

In [ ]:
result_6bii_cost = ...
result_6bii_timing = ...

In [ ]:
grader.check("6bii")

## Question 6c
Now, let's see the impact of adding an index on the `g_batting` column. Create an index named `appearances_g_batting_idx` on the `g_batting` column. Re-inspect the queries from `Question 6a` and `Question 6b` and record the respective execution costs and times.

In [ ]:
%%sql
DROP INDEX IF EXISTS appearances_g_batting_idx;
DROP INDEX IF EXISTS appearances_g_batting_all_idx;
...

In [ ]:
# record the updated costs for Question 6a ("and" query)
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_6ai}";

In [ ]:
result_6c_and_index_cost = ...
result_6c_and_index_timing = ...

In [ ]:
# record the updated costs for Question 6b ("or" query)
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_6bi}";

In [ ]:
result_6c_or_index_cost = ...
result_6c_or_index_timing = ...

In [ ]:
grader.check("q6c")

<br/><br/>

---

## Question 6d: Multiple-attribute index

Now, create a multiple column index on `g_batting` and `g_all` called `appearances_g_batting_all_idx` and record the query execution time and cost for the "or" command in `Question 6b`.

Before continuing, make sure to drop all existing indexes from previous questions.

In [ ]:
# you must run this cell!!!
%sql DROP INDEX IF EXISTS appearances_g_batting_idx;
%sql DROP INDEX IF EXISTS salary_idx;

In [ ]:
%%sql
DROP INDEX IF EXISTS appearances_g_batting_all_idx;
...

In [ ]:
# record the updated costs for Question 6b ("or" query)
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_6bi}";

In [ ]:
result_6d_multiple_col_index_cost = ...
result_6d_multiple_col_index_timing = ...

In [ ]:
grader.check("q6d")

<br/><br/>

---

## Question 6e
Given your findings from inspecting the query plans from all parts of this `Question 6`, assign the variable `q6e` to a list of **ALL** below statements that are true.

A. Adding an index on a column used in an `AND` predicate will reduce the query cost _but not_ the execution time.<br/>
B. Adding an index on a column used in an `AND` predicate will reduce the query cost _and_ the execution time.<br/>
C. Adding an index on a column used in an `OR` predicate will reduce the query cost _but not_ the execution time.<br/>
D. Adding an index on a column used in an `OR` predicate will reduce the query cost _and_ the execution time.<br/>
E. Adding a multicolumn index on columns in an `OR` predicate will reduce the query cost _but not_ the execution time.<br/>
F. Adding a multicolumn index on columns in an `OR` predicate will reduce the query cost _and_ the execution time.

**Note:** Your answer should have the format like this if you think A and B are both true: `q6e = ['A', 'B']`. The autograder is **case sensitive** but order should not matter.

In [ ]:
q6e = ...

In [ ]:
grader.check("q6e")

<!-- BEGIN QUESTION -->

### Question 6ei Justification

1. **Explain your answer to `Question 6e`** above based on your understanding of both the theoretical aspects of query planning and optimization discussed in class, as well as the EXPLAIN ANALYZE results - but bear in mind that sometimes EXPLAIN ANALYZE results may be unpredictable/hard to explain (because of contention, caching, noise/randomness).
2. If there were any options you did NOT select, **choose any one of them and explain why you did not select it.**

Please limit your answer to 60 words (~3 sentences). You are required to **explicitly state which answer option(s) you chose or didn't choose** in addition to your explanations. For example, you could write, "I chose (A) because..." or "I did NOT choose (B) because..."

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/><br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 7: Indexes and Aggregations, Part 3

Continue the analysis on how indexes impact query performance.

Before continuing, make sure to drop all existing indexes from previous questions.

In [ ]:
# you must run this cell!!!
%sql DROP INDEX IF EXISTS appearances_g_batting_idx;
%sql DROP INDEX IF EXISTS salary_idx;
%sql DROP INDEX IF EXISTS appearances_g_batting_all_idx;

---

## Question 7a

Write 2 queries, one that finds the minimum salary from the salary table `salaries` and one that finds the average. Inspect the queries' query plans and record their execution times and costs.

### Question 7ai

**Find the minimum salary.** Call this column `min_salary`.

Your table header should look like this:

| min_salary |
| --- |

In [ ]:
%%sql
/* Make sure to run this cell if you have already run Question 7c 
    and have not restarted your kernel since */
    
DROP INDEX IF EXISTS salary_idx;

In [ ]:
%%sql --save query_7ai result_7ai <<
...

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_7ai = %sqlcmd snippets query_7ai
grading_util.save_results("result_7ai", query_7ai, result_7ai);

display(result_7ai)

!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_7ai}";

In [ ]:
result_7ai_query_cost = ...
result_7ai_query_timing = ...

In [ ]:
grader.check("q7ai")

### Question 7aii

**Find the average salary.** Call this column `average_salary`.

Your table header should look like this:

| average_salary |
| --- |

In [ ]:
%%sql
/* Make sure to run this cell if you have already run Question 7c 
    and have not restarted your kernel since */
    
DROP INDEX IF EXISTS salary_idx;

In [ ]:
%%sql --save query_7aii result_7aii <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_7aii = %sqlcmd snippets query_7aii
grading_util.save_results("result_7aii", query_7aii, result_7aii);

display(result_7aii)

!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_7aii}";

In [ ]:
result_7aii_query_cost = ...
result_7aii_query_timing = ...

In [ ]:
grader.check("q7aii")

<br><br>

---
## Question 7b
Create an index on the `salary` column in the `salaries` table and re-inspect the query plans from the previous part and record the respective execution time and cost.

In [ ]:
%%sql
DROP INDEX IF EXISTS salary_idx;
...

In [ ]:
# record the updated costs for "min" query
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_7ai}";

In [ ]:
result_7b_min_query_cost = ...
result_7b_min_query_timing = ...

In [ ]:
# record the updated costs for "avg" query
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_7aii}";

In [ ]:
result_7b_avg_query_cost = ...
result_7b_avg_query_timing = ...

In [ ]:
grader.check("q7b")

<!-- BEGIN QUESTION -->

<br><br>

---

## Question 7c
Given your findings from `Question 7`, which of the following statements is true? **Select one.**

<br> A. An index on the column being aggregated in a query will always provide a performance enhancement.
<br> B. A query finding the `MIN(salary)` will always benefit from an index on salary, but a query finding `MAX(salary)` will not.
<br> C. A query finding the `COUNT(salary)` will always benefit from an index on salary, but a query finding `AVG(salary)` will not.
<br> D. Queries finding the `MIN(salary)` or `MAX(salary)` will always benefit from an index on salary, but queries finding `AVG(salary)` or `COUNT(salary)` will not.

**State and justify your answer.**

1. **Explain your answer** based on your understanding of both the theoretical aspects of query planning and optimization discussed in class, as well as the EXPLAIN ANALYZE results - but bear in mind that sometimes EXPLAIN ANALYZE results may be unpredictable/hard to explain (because of contention, caching, noise/randomness).
2. **Of the answer options you did not select, choose any one of them and explain why that option is wrong.**

Please limit your response to 60 words (~3 sentences).
 
**Note:** Your answer should be formatted as follows: `(A) because ...` and `Not (A) because ...`

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/><br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 8: Clustered Indexes
In this question, we will inspect the impact that clustering our data on an index can have on a query's performance.

Before continuing, make sure to drop all existing indexes from previous questions.

In [ ]:
# you must run this cell!!!
%sql DROP INDEX IF EXISTS appearances_g_batting_idx;
%sql DROP INDEX IF EXISTS salary_idx;
%sql DROP INDEX IF EXISTS appearances_g_batting_all_idx;

---

## Question 8a

### Question 8ai

Write a query that finds the `player_id`, `year_id`, `team_id`, and `ab` for all players whose `ab` was above 500. Your table header should look like this:

| player_id | year_id | team_id | ab |
| --- | --- | --- | --- |

Optional: AB, short for ["At bat"](https://en.wikipedia.org/wiki/At_bat), is a baseball player statistic.

In [ ]:
%%sql --save query_8ai result_8ai <<
...
;

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_8ai = %sqlcmd snippets query_8ai
grading_util.save_results("result_8ai", query_8ai, result_8ai);
result_8ai.DataFrame().head(3)

In [ ]:
grader.check("q8ai")

### Question 8aii

Inspect the query plan and record the execution time and cost.

In [ ]:
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_8ai}";

In [ ]:
result_8aii_cost = ...
result_8aii_timing = ...

In [ ]:
grader.check("8aii")

<br><br>

---

## Question 8b

Cluster the `batting` table on its primary key (Hint: use the psql meta-command `\d batting` to find out what name of the primary key is). We are able to directly cluster on the primary key (without first creating a separate index) because Postgres automatically creates an index for it.

Then, re-inspect the query plan for the query from `Question 8a` and record the execution time and cost.

In [ ]:
%%sql
...

In [ ]:
# check the updated costs for query in Question 8a
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_8ai}";

In [ ]:
result_8b_cost = ...
result_8b_timing = ...

In [ ]:
grader.check("q8b")

<br><br>

---

## Question 8c

Now, let's try clustering the table based on another index. Create an index on `ab` column called `ab_idx` in the `batting` table AND cluster the `batting` table with this new index. Now, re-inspect the query plan and record the execution time and cost.

In [ ]:
%%sql --save query_8c result_8c <<
DROP INDEX IF EXISTS ab_idx;
...

In [ ]:
# Do not delete/edit this cell!
# You must run this cell before running the autograder.
query_8c = %sqlcmd snippets query_8c
grading_util.save_results("result_8c", query_8c, result_8c);

# check the updated costs for query in Question 8a
!psql -h localhost -d baseball -c "EXPLAIN ANALYZE {query_8ai}";

In [ ]:
result_8c_cost = ...
result_8c_timing = ...

In [ ]:
grader.check("q8c")

<br><br>

---

## Question 8d
Given your findings from inspecting the query plans from Questions 8a, 8b, and 8c, assign the variable `q8d` to a list of **ALL** statements that are true.

A. Clustering based on the `ab_idx` decreased the cost of the query.<br/>
B. Clustering based on the `ab_idx` increased the cost of the query.<br/>
C. Clustering based on the `ab_idx` increased the execution time of the query.<br/>
D. Clustering based on the `ab_idx` decreased the execution time of the query.<br/>
E. Clustering based on the `batting_pkey` decreased the cost of the query.<br/>
F. Clustering based on the `batting_pkey` increased the cost of the query.<br/>
G. Clustering based on the `batting_pkey` increased the execution time of the query.<br/>
H. Clustering based on the `batting_pkey` decreased the execution time of the query.<br/>
I. None of the above
    
**Note:** Your answer should have the format like this if you think A and B are both true: `q8d = ['A', 'B']`. The autograder is **case sensitive** but order should not matter.

In [ ]:
q8d = ...

In [ ]:
grader.check("q8d")

<!-- BEGIN QUESTION -->

<br><br>

---

### Question 8di Justification

1. **Explain your answer to `Question 8d`** above based on your understanding of both the theoretical aspects of query planning and optimization discussed in class, as well as the EXPLAIN ANALYZE results - but bear in mind that sometimes EXPLAIN ANALYZE results may be unpredictable/hard to explain (because of contention, caching, noise/randomness).
2. If there were any options you did NOT select, **choose any one of them and explain why you did not select it.**

Please limit your answer to **60 words (~3 sentences)**. You are required to **explicitly state which answer option(s) you chose or didn't choose** in addition to your explanations. For example, you could write, "I chose (A) because..." or "I did NOT choose (B) because..."

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<br/><br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 9: Cost of Index Management
Until now, we have seen the positive potential impact that indexes can have on query performance, but remember in real world technologies/applications, we will be routinely receiving new data (and in large quantities) which would trigger regular updates to our tables. In this section, we will dive into the cost of managing the indexes that we create.

Before starting this question, be sure to delete any indexes by running the below cell.

In [ ]:
# you must run this cell!!!
%sql DROP INDEX IF EXISTS appearances_g_batting_idx;
%sql DROP INDEX IF EXISTS salary_idx;
%sql DROP INDEX IF EXISTS appearances_g_batting_all_idx;
%sql DROP INDEX IF EXISTS ab_idx;

---

## Question 9a

Record the time it takes to insert 300,000 rows into the `salaries` table when no additional index is configured.

Run the following cell to setup a column to track which rows we added as part of these inserts.

In [ ]:
%sql ALTER TABLE salaries ADD added boolean DEFAULT False;

Next, run the provided update script and record the **wall time** (found in the 2nd line of output).

**NOTE:** Running the below cell multiple times may result in an error, unless you first delete the rows with the cell given at the end of this subpart.

In [ ]:
%%time
%%sql
DO $$
 DECLARE counter INTEGER := 1;
 BEGIN
     FOR counter IN 100001..400000 LOOP
     INSERT INTO salaries (year_id, team_id, lg_id, player_id, salary, added)
         VALUES (2021, 'ATL', 'NL', 'p' || counter, RANDOM() * 1000000, true);
     END LOOP;
END;
$$;

In [ ]:
result_9a_timing = ...

In [ ]:
grader.check("q9a")

<br/>

**Before moving onto the next question**,  delete all the rows that were added to the table from the update script.

In [ ]:
%%sql
/* just run this cell */
DELETE FROM salaries
WHERE added = 'true';

<br><br>

---

## Question 9b

Now, create an index on the `salary` column and record the **wall time** after executing the update script. Make sure to first run the previous cell to rollback any changes from the previous part!

In [ ]:
%%sql
DROP INDEX IF EXISTS salary_idx;
...

**NOTE:** Running the below cell multiple times may result in an error, unless you first delete the rows with the cell given at the end of last subpart.

In [ ]:
%%time
%%sql
DO $$
 DECLARE counter INTEGER := 1;
 BEGIN
     FOR counter IN 100001..400000 LOOP
     INSERT INTO salaries (year_id, team_id, lg_id, player_id, salary, added)
         VALUES (2021, 'ATL', 'NL', 'p' || counter, RANDOM() * 1000000, true);
     END LOOP;
END;
$$;

In [ ]:
result_9b_timing = ...

In [ ]:
grader.check("q9b")

<!-- BEGIN QUESTION -->

<br><br>

---

## Question 9c
What difference did you notice when you added an index into the salaries table and re-timed the update? Why do you think it happened? Please limit your response to 60 words (~3 sentences).

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

<br/><br/><br/>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Question 10: Project Takeaways

In this project, we explored how the database system optimizes query execution and how users can futher tune the performance of their queries.

Familiarizing yourself with these optimization and tuning methods will make you a better data engineer. In this question, we'll ask you to recall and summarize these concepts. Who knows? Maybe one day it will help you during an interview or on a project.

In the following answer cell,
1. Name 3 non-trivial, **distinctly different** methods to optimize query performance that you learned in this project. Some places to start looking are some of the question topics.
2. For each method, summarize how and why it can optimize query performance. Feel free to discuss any drawbacks, if applicable.

Please limit your response to **200 words (~10 sentences)**. Each method identification/discussion is 2 points.


_Type your answer here, replacing this text._

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# Congratulations! You have finished Project 2.

Run the following cell to zip and download the results of your queries. You will also need to run the export cell at the end of the notebook.

**Please save your notebook before exporting (this is a good time to do it!)** Otherwise, we may not be able to export your written responses to `proj2.pdf`. We will not be accepting regrade requests for failure to render written responses.

**For your submission on Gradescope, you will only need to submit the single `proj2.zip` file generated by the export cell.** Please ensure that your submission `proj2.zip` file includes `proj2.pdf`, `proj2.ipynb`, and `results.zip`.  You do not need to submit anything separately to the written submission page on Gradescope.

**Please ensure that public tests pass upon submission.** It is your responsibility to wait until the autograder finishes running. We will not be accepting regrade requests for submission issues.

**Common submission issues:** You MUST submit the generated zip file to the autograder. However, Safari is known to automatically unzip files upon downloading. You can fix this by going into Safari preferences, and deselect the box with the text "Open safe files after downloading" under the "General" tab. If you experience issues with downloading via clicking on the link, you can also navigate to the project 2 directory within JupyterHub (remove `proj2.ipynb` from the url), and manually download the generated zip files. Please post on Ed if you encounter any other submission issues.

Run the following cell to zip and download the results of your queries. You will also need to run the export cell at the end of the notebook.

In [ ]:
grading_util.prepare_submission_and_cleanup()

In [ ]:
# Close SQL magic connection
# You may disregard "RunTimeError: Could not close connection"
# %sql --close postgresql://127.0.0.1:5432/baseball

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(files=['results.zip'])